# DataevalShift Analytics Store Demo

This notebook demonstrates the end-to-end workflow for the DataevalShift capability with the analytics store:

1. Load two datasets (reference and evaluation)
2. Run the shift capability (drift detection + OOD analysis)
3. Extract scalar records
4. Write to the analytics store
5. Query results via SQL

## Setup

In [ ]:
import tempfile
from pathlib import Path

from checkmaite.core.analytics_store import AnalyticsStore, ParquetBackend
from checkmaite.core.object_detection import DataevalShift
from checkmaite.core.object_detection.dataset_loaders import CocoDetectionDataset

## Load Datasets

Shift detection requires **two datasets**: a reference dataset and an evaluation dataset.
For this demo we use the same COCO test data as both reference and evaluation,
so we expect no drift to be detected.

In [ ]:
data_root = Path("../../..")
coco_dir = data_root / "tests" / "data_for_tests" / "coco_resized_val2017"

reference_dataset = CocoDetectionDataset(
    root=str(coco_dir),
    ann_file=str(coco_dir / "instances_val2017_resized_6.json"),
    dataset_id="coco-reference",
)

evaluation_dataset = CocoDetectionDataset(
    root=str(coco_dir),
    ann_file=str(coco_dir / "instances_val2017_resized_6.json"),
    dataset_id="coco-evaluation",
)

print(f"Reference dataset: {reference_dataset.metadata['id']} ({len(reference_dataset)} images)")
print(f"Evaluation dataset: {evaluation_dataset.metadata['id']} ({len(evaluation_dataset)} images)")

## Run DataevalShift Capability

The shift capability runs three drift detection tests and out-of-distribution (OOD) analysis:

- **MMD** (Maximum Mean Discrepancy): Compares embedding distributions
- **CVM** (Cramer-von Mises): Per-feature univariate distribution test
- **KS** (Kolmogorov-Smirnov): Per-feature distribution test
- **OOD KNN**: Flags individual evaluation samples that are out-of-distribution

In [ ]:
capability = DataevalShift()
run = capability.run(datasets=[reference_dataset, evaluation_dataset], use_cache=False)

print(f"Run UID: {run.run_uid[:24]}...")
print(f"\n--- Drift Detection ---")
print(f"  MMD: drifted={run.outputs.drift.mmd.drifted}, distance={run.outputs.drift.mmd.distance:.4f}, p_val={run.outputs.drift.mmd.details['p_val']:.4f}")
print(f"  CVM: drifted={run.outputs.drift.cvm.drifted}, distance={run.outputs.drift.cvm.distance:.4f}, p_val={run.outputs.drift.cvm.details['p_val']:.4f}")
print(f"  KS:  drifted={run.outputs.drift.ks.drifted}, distance={run.outputs.drift.ks.distance:.4f}, p_val={run.outputs.drift.ks.details['p_val']:.4f}")
print(f"\n--- OOD Detection ---")
import numpy as np
is_ood = run.outputs.ood.ood_knn.is_ood
print(f"  OOD count: {int(np.sum(is_ood))} / {len(is_ood)}")
print(f"  Mean instance score: {float(np.mean(run.outputs.ood.ood_knn.instance_score)):.4f}")
print(f"  Std instance score: {float(np.std(run.outputs.ood.ood_knn.instance_score)):.4f}")
print(f"  Max instance score: {float(np.max(run.outputs.ood.ood_knn.instance_score)):.4f}")

## Extract Records

The `extract()` method distills the shift outputs into flat scalar records.
Drift tests produce 12 scalar fields (3 tests x 4 values each: drifted, distance, p_val, threshold).
OOD per-sample arrays are aggregated into count, total, ratio, and mean score.

In [ ]:
records = run.extract()
print(f"Extracted {len(records)} record(s)\n")

record = records[0]
print(f"Table: {record.table_name}")
print(f"Reference dataset: {record.reference_dataset_id}")
print(f"Evaluation dataset: {record.evaluation_dataset_id}")
print(f"\n--- Drift ---")
print(f"  MMD: drifted={record.mmd_drifted}, distance={record.mmd_distance:.4f}, p_val={record.mmd_p_val:.4f}, threshold={record.mmd_threshold}")
print(f"  CVM: drifted={record.cvm_drifted}, distance={record.cvm_distance:.4f}, p_val={record.cvm_p_val:.4f}, threshold={record.cvm_threshold}")
print(f"  KS:  drifted={record.ks_drifted}, distance={record.ks_distance:.4f}, p_val={record.ks_p_val:.4f}, threshold={record.ks_threshold}")
print(f"\n--- OOD ---")
print(f"  Count: {record.ood_count} / {record.ood_total}")
print(f"  Ratio: {record.ood_ratio:.4f}")
print(f"  Mean instance score: {record.ood_mean_instance_score:.4f}")
print(f"  Std instance score: {record.ood_std_instance_score:.4f}")
print(f"  Max instance score: {record.ood_max_instance_score:.4f}")
print(f"\n--- Per-Feature Drift ---")
print(f"  CVM features drifted: {record.cvm_feature_drift_count}")
print(f"  KS features drifted: {record.ks_feature_drift_count}")

## Write to Analytics Store

In [ ]:
store_dir = tempfile.mkdtemp(prefix="shift_demo_")
store = AnalyticsStore(ParquetBackend(store_dir))

store.write([run])

print(f"Store path: {store_dir}")
print(f"Tables: {store.list_tables()}")
print(f"\nShift table schema: {store.describe_table('dataeval_shift')}")

## Query via SQL

All results are now queryable via standard SQL.

In [ ]:
# View all shift records
df = store.query_sql("SELECT * FROM dataeval_shift")
print(df)

In [ ]:
# Query drift results only
df = store.query_sql("""
    SELECT
        reference_dataset_id,
        evaluation_dataset_id,
        mmd_drifted,
        mmd_p_val,
        mmd_threshold,
        cvm_drifted,
        cvm_p_val,
        cvm_threshold,
        ks_drifted,
        ks_p_val,
        ks_threshold,
        cvm_feature_drift_count,
        ks_feature_drift_count
    FROM dataeval_shift
""")
print(df)

In [ ]:
# Check the auto-populated runs table
df_runs = store.query_sql("""
    SELECT run_uid, capability_table, entity_type, entity_id
    FROM runs
""")
print(df_runs)

## Cross-Capability JOIN Example

Shift uses two dataset IDs (`reference_dataset_id` and `evaluation_dataset_id`)
instead of the single `dataset_id` used by other capabilities. You can JOIN on
either side. For example, to correlate the reference dataset's feasibility with
drift results:

```sql
SELECT
    s.reference_dataset_id,
    s.mmd_drifted,
    s.mmd_p_val,
    f.ber_upper
FROM dataeval_shift s
JOIN dataeval_feasibility f ON s.reference_dataset_id = f.dataset_id
```